Databricks notebook source
SENSE 프로젝트 — Silver Layer: FRED raw → silver
담당: 1조 (정형 데이터)

목적:
  ADLS Gen2 raw/fred_macro_data/ 에서 FRED 금리/매크로 지표를 읽어
  master_calendar 기반 시계열 정제 후 silver/fred/ 에 저장

처리 대상 지표 (지표코드):
  DGS10       : 미국 10년물 국채 금리
  DGS2        : 미국 2년물 국채 금리
  T10Y2Y      : 장단기 금리차 (DGS10 - DGS2) ← 경기침체 선행지표
  DFF         : 연준 기준금리 (Federal Funds Rate)
  DFII10      : 미국 10년물 실질금리 (물가 반영)
  BAMLH0A0HYM2: 하이일드 스프레드 ← 시장 공포 지수

[파생 컬럼]
  yield_spread      : 장단기 금리차 변화율 (T10Y2Y 일별 변동)
  stagnation_pressure : 스태그플레이션 압력 지수 (금리 + 실질금리 결합)

[FRED 데이터 특성]
  - 미국 영업일 기준 발표 (주말·미국 공휴일은 결측)
  - master_calendar 로 미국 영업일 필터 후 Forward Fill 적용


# 0. 스토리지 계정 설정 및 ADLS OAuth 인증


In [0]:
# =======================================================
# 스토리지 계정 및 경로 상수 정의
# =======================================================
STORAGE_ACCOUNT  = "3dtteam1adls"
SECRET_SCOPE     = "sense-kv"
BRONZE_CONTAINER    = "raw"
SILVER_CONTAINER = "curated"

BASE_PATH_BRONZE    = f"abfss://{BRONZE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
BASE_PATH_SILVER = f"abfss://{SILVER_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

# master_calendar 경로
CALENDAR_PATH = f"{BASE_PATH_SILVER}/master_calendar.parquet"

# ============================================================
# ADLS Gen2 OAuth 인증 (Service Principal)
# ============================================================
spark.conf.set(
    f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    dbutils.secrets.get(scope=SECRET_SCOPE, key="adls-client-id")
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    dbutils.secrets.get(scope=SECRET_SCOPE, key="adls-client-secret")
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{dbutils.secrets.get(scope=SECRET_SCOPE, key='adls-tenant-id')}/oauth2/token"
)

print(f"✅ ADLS OAuth 인증 설정 완료: {STORAGE_ACCOUNT}")


# 1. master_calendar 로드

FRED는 **미국 영업일 기준**으로만 값이 존재합니다.
캘린더의 `미국_휴장일_여부` 컬럼을 활용해 두 가지 처리를 합니다.

1. **미국 영업일만 유효 행으로 인정** (미국 공휴일 = 실제 결측)
2. **Forward Fill 기준 제공** (미국 공휴일 날짜는 직전 영업일 값으로 채움)


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import to_date, col

# master_calendar 로드
calendar_df = (
    spark.read
    .parquet(CALENDAR_PATH)
    .withColumn("기준일자", to_date(col("기준일자"), "yyyy-MM-dd"))
    .select("기준일자", "주말여부", "한국_휴장일_여부", "미국_휴장일_여부")
)

# 미국 실제 영업일 (주말 X, 미국 공휴일 X)
us_biz_days = (
    calendar_df
    .filter(col("미국_휴장일_여부") == False)
    .select(col("기준일자").alias("us_biz_date"))
)

print(f"✅ master_calendar 로드 완료")
print(f"   전체 기간    : {calendar_df.count()}일")
print(f"   미국 영업일  : {us_biz_days.count()}일")
print(f"   미국만 휴장  : {calendar_df.filter((col('미국_휴장일_여부')==True) & (col('주말여부')==False)).count()}건")
display(calendar_df.limit(10))


# 2. FRED raw 데이터 로드

raw 경로 구조:
```
raw/fred_macro_data/
  └── rates_data_250408-260406.csv
```

실제 컬럼 구조:
| 컬럼 | 타입 | 예시 |
|---|---|---|
| `관측일자` | string | 2026-04-03 |
| `지표코드` | string | DGS10 |
| `금리값` | double | 4.35 |
| `수집일시` | string | 2026-04-07 06:45:46 |

> ⚠️ FRED raw는 **CSV 포맷**입니다.
> `pathGlobFilter`로 `.csv` 파일만 명시적으로 로드합니다.


In [0]:
fred_raw_경로 = f"{BASE_PATH_BRONZE}/fred_macro_data"

fred_raw_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("recursiveFileLookup", "true")
    .option("pathGlobFilter", "*.csv")
    .option("encoding", "utf-8")
    .option("inferSchema", "true")
    .load(fred_raw_경로)
)

print(f"✅ FRED raw 로드 완료: {fred_raw_df.count()}행")
print(f"   컬럼: {fred_raw_df.columns}")
print(f"\n지표코드별 건수:")
display(fred_raw_df.groupBy("지표코드").count().orderBy("지표코드"))
display(fred_raw_df.limit(10))


# 3. 스키마 정리

- `관측일자` → `date` (DateType)
- `지표코드` → `indicator` (영문 컬럼명 통일)
- `금리값` → `value` (영문 컬럼명 통일)
- `수집일시` 컬럼 제거 (분석 불필요)
- 중복 제거: (`date`, `indicator`) 기준
- null 제거: `date`, `value`


In [0]:
from pyspark.sql.functions import to_date, year, month, col

fred_clean_df = (
    fred_raw_df
    .withColumn("date",      to_date(col("관측일자"), "yyyy-MM-dd"))
    .withColumnRenamed("지표코드", "indicator")
    .withColumnRenamed("금리값",   "value")
    .drop("관측일자", "수집일시")
    .dropDuplicates(["date", "indicator"])
    .filter(col("date").isNotNull())
    .filter(col("value").isNotNull())
    .orderBy("indicator", "date")
)

print(f"✅ 스키마 정리 완료: {fred_clean_df.count()}행")
fred_clean_df.printSchema()
display(fred_clean_df.limit(10))


# 4. Forward Fill (미국 영업일 기준 결측치 보간)

FRED 지표는 미국 영업일에만 값이 발표됩니다.
미국 공휴일(예: 독립기념일, MLK Day)에는 값이 없어서 그냥 두면 Gold JOIN 시 null이 됩니다.

### 처리 방식

```
미국 영업일 캘린더(us_biz_days) 기준으로 날짜 뼈대를 만들고
실제 값이 있는 날은 그대로 사용, 없는 날은 직전 값으로 채움
```

> 예시: 2025-01-20 (MLK Day, 미국 공휴일)
> → 직전 영업일 2025-01-17 의 DGS10 값 4.62 로 채움

### 왜 이렇게 하나?
금리는 공휴일이라고 갑자기 변하지 않습니다.
전날 값을 그대로 유지하는 게 현실을 가장 잘 반영합니다.


In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import last

# 처리 대상 지표 목록
FRED_INDICATORS = ["DGS10", "DGS2", "T10Y2Y", "DFF", "DFII10", "BAMLH0A0HYM2"]

# ── 지표별 Wide 포맷 피벗 ────────────────────────────────────────────
# Long → Wide: date × indicator → 컬럼
fred_wide_df = (
    fred_clean_df
    .filter(col("indicator").isin(FRED_INDICATORS))
    .groupBy("date")
    .pivot("indicator", FRED_INDICATORS)
    .agg(F.first("value"))
)

# ── 미국 영업일 캘린더를 기준 뼈대로 사용 ────────────────────────────
# 캘린더 날짜 범위와 fred 데이터 범위를 맞춤
fred_date_range = fred_wide_df.agg(
    F.min("date").alias("min_date"),
    F.max("date").alias("max_date")
).collect()[0]

calendar_base = (
    us_biz_days
    .filter(
        (col("us_biz_date") >= fred_date_range["min_date"]) &
        (col("us_biz_date") <= fred_date_range["max_date"])
    )
    .withColumnRenamed("us_biz_date", "date")
)

# ── 캘린더 뼈대에 FRED 실제값 LEFT JOIN ───────────────────────────────
fred_joined_df = calendar_base.join(fred_wide_df, on="date", how="left")

# ── Forward Fill (지표별로 적용) ──────────────────────────────────────
w_ffill = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, 0)

fred_ffill_df = fred_joined_df
for indicator in FRED_INDICATORS:
    fred_ffill_df = fred_ffill_df.withColumn(
        indicator,
        last(col(indicator), ignorenulls=True).over(w_ffill)
    )

print(f"✅ Forward Fill 완료: {fred_ffill_df.count()}행")
display(fred_ffill_df.orderBy("date").limit(15))


# 5. 파생 컬럼 생성

| 컬럼명 | 계산 방법 | 의미 |
|---|---|---|
| `yield_spread` | DGS10 - DGS2 | 장단기 금리차. 음수면 경기침체 신호 |
| `yield_spread_change` | 전일 대비 yield_spread 변화량 | 금리 커브 변화 속도 |
| `stagnation_pressure` | DGS10 + DFII10 의 정규화 결합 | 스태그플레이션 압력 지수 |

### 쉬운 설명

**yield_spread (장단기 금리차)**
> 은행에서 10년 만기 예금 금리와 2년 만기 예금 금리 차이입니다.
> 보통은 10년짜리가 더 높아야 정상인데,
> 이게 역전(음수)되면 "경기가 나빠질 것 같다"는 시장의 신호입니다.

**stagnation_pressure (스태그플레이션 압력)**
> 경제가 불황인데 물가까지 오르는 최악의 상황을 수치화한 것입니다.
> 명목금리(DGS10)와 실질금리(DFII10)가 동시에 높으면 압력이 올라갑니다.


In [0]:
from pyspark.sql.functions import lag, when, lit
from pyspark.sql import Window

w_date = Window.orderBy("date")

fred_featured_df = (
    fred_ffill_df

    # ── 장단기 금리차 ────────────────────────────────────────────────
    .withColumn(
        "yield_spread",
        col("DGS10") - col("DGS2")
    )

    # ── 장단기 금리차 일별 변화량 ────────────────────────────────────
    .withColumn(
        "yield_spread_change",
        col("yield_spread") - lag("yield_spread", 1).over(w_date)
    )

    # ── 스태그플레이션 압력 지수 ─────────────────────────────────────
    # DGS10 (명목금리) 과 DFII10 (실질금리) 단순 합산
    # 두 지표가 모두 높을수록 압력이 강하다고 판단
    .withColumn(
        "stagnation_pressure",
        when(
            col("DGS10").isNotNull() & col("DFII10").isNotNull(),
            col("DGS10") + col("DFII10")
        ).otherwise(lit(None))
    )

    # ── 첫 행 null → 0 채우기 (lag 초기값 부재) ─────────────────────
    .na.fill(0.0, subset=["yield_spread_change"])

    # ── 파티션 컬럼 추가 ────────────────────────────────────────────
    .withColumn("year",  F.year("date"))
    .withColumn("month", F.month("date"))
)

print("✅ 파생 컬럼 생성 완료")
display(
    fred_featured_df
    .select("date", "DGS10", "DGS2", "yield_spread",
            "yield_spread_change", "stagnation_pressure")
    .orderBy("date")
    .limit(15)
)

# 6. 데이터 검증


In [0]:
# ── 날짜 범위 확인 ──────────────────────────────────────────────────
date_range = fred_featured_df.agg(
    F.min("date").alias("start_date"),
    F.max("date").alias("end_date"),
    F.count("date").alias("영업일수")
).collect()[0]

print("=== 📅 날짜 범위 ===")
print(f"  시작: {date_range['start_date']}")
print(f"  종료: {date_range['end_date']}")
print(f"  미국 영업일수: {date_range['영업일수']}일")

# ── 지표별 null 비율 확인 ────────────────────────────────────────────
print("\n=== 🔍 지표별 null 비율 ===")
total = fred_featured_df.count()
check_cols = FRED_INDICATORS + ["yield_spread", "stagnation_pressure"]
for c in check_cols:
    null_cnt = fred_featured_df.filter(col(c).isNull()).count()
    status = "✅" if null_cnt == 0 else "⚠️"
    print(f"  {status} {c}: null {null_cnt}건 ({null_cnt/total*100:.1f}%)")

# ── yield_spread 음수 구간 확인 (경기침체 신호 날짜) ─────────────────
print("\n=== 📉 yield_spread 역전 구간 (경기침체 신호) ===")
inversion = fred_featured_df.filter(col("yield_spread") < 0)
print(f"  역전 발생일수: {inversion.count()}일")
display(inversion.select("date", "DGS10", "DGS2", "yield_spread").orderBy("date").limit(10))


# 7. Silver 저장

- 포맷: **Parquet** (기존 노트북 방식 유지)
- 파티션: `year` / `month`
- 모드: `overwrite`

> ⚠️ overwrite 사용 이유:
> `yield_spread_change` 등 파생 컬럼이 전체 시계열 참조 → 매번 전체 재처리


In [0]:
SILVER_PATH = f"{BASE_PATH_SILVER}/fred"

(
    fred_featured_df
    .write
    .mode("overwrite")
    .partitionBy("year", "month")
    .parquet(SILVER_PATH)
)

saved_files = dbutils.fs.ls(SILVER_PATH)
print(f"✅ Silver 저장 완료")
print(f"   경로     : {SILVER_PATH}")
print(f"   파티션 수: {len(saved_files)}개")
for f in saved_files:
    print(f"   - {f.name}")


# 8. 저장 확인 (sanity check)


In [0]:
df_check = spark.read.parquet(SILVER_PATH)

print(f"✅ silver 읽기 확인 — 행수: {df_check.count():,}")
print(f"   컬럼: {df_check.columns}")
display(df_check.orderBy("date").limit(10))


## ✅ 완료 후 다음 단계

| 순서 | 노트북 | 상태 |
|---|---|---|
| 1 | `01_yfinance_raw_to_silver.py` | ✅ 완료 |
| 2 | `02_fred_raw_to_silver.py` | ✅ 완료 |
| 3 | `03_fx_raw_to_silver.py` | ⏳ 대기 |
| 4 | `04_silver_to_gold.py` | ⏳ 대기 |

> **Gold JOIN 시 참고:**
> - FRED silver는 `date` = 미국 영업일 기준
> - yfinance silver는 `effective_kr_date` 기준
> - `04_silver_to_gold` 에서 날짜 컬럼 통일 후 JOIN 필요
